# Test Agents Locally

Start agents locally using Kagenti ADK and verify they expose proper A2A endpoints.

## 1. Load Environment

In [ ]:
import os
from dotenv import load_dotenv

env_path = os.path.join(os.path.dirname(os.getcwd()), ".env")
load_dotenv(env_path)
print("✅ Environment loaded")

## 2. Start Agents

Start agents in the background. In practice, run `make agents-start` from the terminal.

In [ ]:
import subprocess
import time

print("Starting agents... (run 'make agents-start' in terminal if not already running)")
print("")
print("Expected agent endpoints:")
print("  Orchestrator:     http://localhost:8100")
print("  Doc Processor:    http://localhost:8101")
print("  Researcher:       http://localhost:8102")
print("  Writer:           http://localhost:8103")
print("  Reviewer:         http://localhost:8104")
print("")
print("⚠️ Make sure agents are running before proceeding")

## 3. Verify A2A AgentCard Endpoints

Each agent should expose `/.well-known/agent-card.json` for A2A discovery.

In [ ]:
import httpx

agents = {
    "orchestrator": "http://localhost:8100",
    "doc_processor": "http://localhost:8101",
    "researcher": "http://localhost:8102",
    "writer": "http://localhost:8103",
    "reviewer": "http://localhost:8104",
}

for name, url in agents.items():
    try:
        resp = httpx.get(f"{url}/.well-known/agent-card.json", timeout=5)
        if resp.status_code == 200:
            card = resp.json()
            skills = [s.get("name", "") for s in card.get("skills", [])]
            print(f"  ✅ {name}: {card.get('name', 'N/A')} — skills: {skills}")
        else:
            print(f"  ❌ {name}: HTTP {resp.status_code}")
    except Exception as e:
        print(f"  ❌ {name}: {e}")

print("\nAgent discovery check complete")

## 4. Send Test Message via A2A

Send a message to the Researcher agent using the A2A JSON-RPC protocol.

In [ ]:
import json

payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": "test-001",
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": "What is document processing?"}],
        }
    },
}

try:
    resp = httpx.post("http://localhost:8102", json=payload, timeout=60)
    result = resp.json()
    print("A2A Response from Researcher:")
    print(json.dumps(result, indent=2)[:1000])
    print("\n✅ A2A communication working")
except Exception as e:
    print(f"❌ Error: {e}")
    print("   Make sure the researcher agent is running on port 8102")

## 5. Test Writer Agent via A2A

In [ ]:
payload = {
    "jsonrpc": "2.0",
    "method": "message/send",
    "id": "test-002",
    "params": {
        "message": {
            "role": "user",
            "parts": [{
                "kind": "text",
                "text": "Write a brief summary about multi-agent systems and their benefits."
            }],
        }
    },
}

try:
    resp = httpx.post("http://localhost:8103", json=payload, timeout=60)
    result = resp.json()
    print("A2A Response from Writer:")
    # Extract text from response
    if "result" in result:
        artifacts = result["result"].get("artifacts", [])
        for art in artifacts:
            for part in art.get("parts", []):
                if part.get("kind") == "text":
                    print(part["text"][:500])
    print("\n✅ Writer agent responding")
except Exception as e:
    print(f"❌ Error: {e}")